# Understanding Area and Volume Estimation with `skimage.measure.regionprops`

Welcome to this interactive bioimage analysis tutorial! 

When we measure structures in biology (like cell area or nuclear volume), we are working with digitized representations of continuous objects. In `scikit-image`, the `regionprops` function calculates the **`area`** property by essentially counting the number of pixels (in 2D) or voxels (in 3D) belonging to a labeled region. 

In this notebook, we will explore two critical concepts:
1. **Digitization and Angles (2D):** How rasterization (representing shapes on a square grid) and rotation alter the pixel count of squares and disks.
2. **Spacing and Volume (3D):** How anisotropic voxel spacing (common in Z-stacks from confocal microscopes) scales our voxel count into physical volume. Note that in `scikit-image`, the property for $N$-dimensional volume is still called `area`.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from skimage.measure import label, regionprops
from skimage.transform import rotate
from skimage.draw import disk
from skimage.morphology import ball, cube
import ipywidgets as widgets
from IPython.display import display

## 1. The 2D World: Rasterization and Angles

In continuous math, rotating a shape does not change its area. However, images are discrete grids of pixels. When we rotate a binary mask of a shape, the interpolation and re-thresholding (rasterization) cause pixels on the boundary to be included or excluded depending on how they align with the grid.

* **Square:** Theoretical area is $L^2$. 
* **Disk:** Theoretical area is $\pi r^2$.

Let's use an interactive slider to rotate a $20 \times 20$ square and a disk (radius $= 10$). Watch how the `regionprops` area fluctuates slightly depending on the angle!

In [2]:
def analyze_2d_area(angle):
    # 1. Create base images (50x50 grid)
    square = np.zeros((50, 50), dtype=np.uint8)
    square[15:35, 15:35] = 1  # 20x20 square (Area = 400)
    
    circle = np.zeros((50, 50), dtype=np.uint8)
    rr, cc = disk((25, 25), 10) # Radius 10 (Area approx 314.16)
    circle[rr, cc] = 1
    
    # 2. Rotate shapes (order=0 means nearest-neighbor interpolation to keep it strictly binary)
    square_rot = rotate(square, angle, order=0, resize=False) > 0.5
    circle_rot = rotate(circle, angle, order=0, resize=False) > 0.5
    
    # 3. Label regions and extract properties
    prop_square = regionprops(label(square_rot))[0]
    prop_circle = regionprops(label(circle_rot))[0]
    
    # 4. Plotting
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    
    axes[0].imshow(square_rot, cmap='gray')
    axes[0].set_title(f"Square\nEstimated Area: {prop_square.area} px\nTheoretical Area: 400 px")
    axes[0].axis('off')
    
    axes[1].imshow(circle_rot, cmap='gray')
    axes[1].set_title(f"Disk\nEstimated Area: {prop_circle.area} px\nTheoretical Area: ~314.16 px")
    axes[1].axis('off')
    
    plt.suptitle(f"Rotation Angle: {angle}°", fontsize=14)
    plt.tight_layout()
    plt.show()

# Create interactive slider
widgets.interact(analyze_2d_area, angle=widgets.IntSlider(min=0, max=90, step=1, value=0));

interactive(children=(IntSlider(value=0, description='angle', max=90), Output()), _dom_classes=('widget-intera…

## 2. The 3D World: Volume and Voxel Spacing

In 3D microscopy, voxels are rarely perfect cubes. Often, the lateral resolution (X and Y) is much higher than the axial resolution (Z). This means a single voxel represents a physical rectangular prism, not a cube. 

By default, `regionprops` just returns the raw voxel count. To get true physical volume, we must pass the `spacing` parameter, which is a tuple representing `(Z_spacing, Y_spacing, X_spacing)`. `scikit-image` will automatically scale the voxel count by the volume of a single voxel:

$$Volume_{physical} = N_{voxels} \times (S_z \times S_y \times S_x)$$

*Note: In `skimage`, the N-dimensional hyper-volume property is still accessed via `.area`.*

Let's test this with a discrete ball (radius $= 10$) and a discrete cube (width $= 21$).

In [ ]:
def analyze_3d_volume(z_spacing, y_spacing, x_spacing):
    # 1. Generate 3D shapes
    img_ball = ball(10)  # Generates a 21x21x21 array
    img_cube = cube(21)  # Generates a 21x21x21 array
    
    # 2. Label regions
    label_ball = label(img_ball)
    label_cube = label(img_cube)
    
    # 3. Calculate spacing and voxel volume
    spacing = (z_spacing, y_spacing, x_spacing)
    voxel_volume = z_spacing * y_spacing * x_spacing
    
    # 4. Measure properties using the spacing parameter
    props_ball = regionprops(label_ball, spacing=spacing)[0]
    props_cube = regionprops(label_cube, spacing=spacing)[0]
    
    # 5. Output Results
    print(f"{'='*40}")
    print(f"SPACING CALIBRATION: {spacing}")
    print(f"Volume of 1 Voxel: {voxel_volume:.3f} physical units³")
    print(f"{'='*40}\n")
    
    print("🟢 DISCRETE BALL (Radius = 10 voxels):")
    print(f"  Voxel Count (Raw): {np.sum(img_ball)}")
    print(f"  Estimated Physical Volume: {props_ball.area:.2f}")
    # Theoretical continuous volume scaled by the discrete grid approximation
    theoretical_ball = (4/3) * np.pi * (10**3) * voxel_volume
    print(f"  Theoretical Continuous Volume: {theoretical_ball:.2f}\n")
    
    print("🟩 DISCRETE CUBE (21x21x21 voxels):")
    print(f"  Voxel Count (Raw): {np.sum(img_cube)}")
    print(f"  Estimated Physical Volume: {props_cube.area:.2f}")
    theoretical_cube = (21**3) * voxel_volume
    print(f"  Theoretical Continuous Volume: {theoretical_cube:.2f}")

# Create interactive sliders for X, Y, and Z spacing
widgets.interact(analyze_3d_volume, 
                 z_spacing=widgets.FloatSlider(min=0.1, max=3.0, step=0.1, value=1.0, description='Z Spacing'),
                 y_spacing=widgets.FloatSlider(min=0.1, max=3.0, step=0.1, value=1.0, description='Y Spacing'),
                 x_spacing=widgets.FloatSlider(min=0.1, max=3.0, step=0.1, value=1.0, description='X Spacing'));